# Cursed Tomb — Persistent Deck Evolution & Solvability Analysis

This notebook analyzes how persistent card degradation (scars, curses, entombment) impacts deck composition and empirical solvability across rounds in *The Cursed Tomb*.

**Key features:**
- Pure core simulation reuse (`sim.deck_evolution_core`) without requiring CLI arguments or `multiprocessing.Pool`.
- Kernel-persistent `runs` state allowing additive overlays across different configurations.
- Interactive widget controls for running and analyzing deck evolution campaigns.

In [2]:
import sys
import os
import random
from pathlib import Path

# Ensure repository root and sim directory are in Python path
repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
sim_dir = str(repo_root / "sim")
if sim_dir not in sys.path:
    sys.path.insert(0, sim_dir)

%matplotlib inline
import matplotlib.pyplot as plt

from sim.deck_evolution_core import (
    run_collapse_campaign,
    aggregate_results,
    plot_evolution,
    write_aggregated_csv,
    load_aggregated_csv,
    RuleFlags,
    DIFFICULTIES,
    HAS_MPL,
)


In [3]:
# Initialize persistent runs list if not already present in kernel environment
if 'runs' not in globals():
    runs = []

def render_plots(show_active=True, show_composition=True, show_solvability=True, *args, **kwargs):
    """Render the time-series figure for all current runs in memory."""
    if not runs:
        print("No runs currently stored in memory. Add a run using widgets or code below.")
        return
    plot_evolution(
        runs,
        show=True,
        show_active=show_active,
        show_composition=show_composition,
        show_solvability=show_solvability,
    )


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    # Ensure render_plots is updated in active kernel session
    def render_plots(show_active=True, show_composition=True, show_solvability=True, *args, **kwargs):
        if not runs:
            print("No runs currently stored in memory. Add a run using widgets or code below.")
            return
        plot_evolution(
            runs,
            show=True,
            show_active=show_active,
            show_composition=show_composition,
            show_solvability=show_solvability,
        )

    style = {'description_width': '140px'}
    layout = widgets.Layout(width='320px')

    difficulty_w = widgets.Dropdown(
        options=list(DIFFICULTIES.keys()),
        value='archaeologist',
        description='Difficulty:',
        style=style,
        layout=layout
    )
    solver_w = widgets.Dropdown(
        options=['greedy', 'heuristic', 'beam', 'dfs'],
        value='greedy',
        description='Solver:',
        style=style,
        layout=layout
    )
    campaigns_w = widgets.BoundedIntText(
        value=10, min=1, max=100, step=1,
        description='Campaigns:',
        style=style,
        layout=layout
    )
    max_rounds_w = widgets.BoundedIntText(
        value=30, min=5, max=5000, step=5,
        description='Max Rounds:',
        style=style,
        layout=layout
    )
    probes_w = widgets.BoundedIntText(
        value=10, min=1, max=100, step=1,
        description='Probes:',
        style=style,
        layout=layout
    )

    # Chart Toggle Checkboxes
    show_active_w = widgets.Checkbox(
        value=True,
        description='Active Cards Chart',
        style=style,
        layout=layout
    )
    show_comp_w = widgets.Checkbox(
        value=True,
        description='Composition Chart',
        style=style,
        layout=layout
    )
    show_solv_w = widgets.Checkbox(
        value=True,
        description='Solvability Chart',
        style=style,
        layout=layout
    )

    btn_add = widgets.Button(description="Add Run", button_style="success", icon="plus", layout=widgets.Layout(width='140px'))
    btn_clear = widgets.Button(description="Clear All Runs", button_style="danger", icon="trash", layout=widgets.Layout(width='140px'))
    btn_update = widgets.Button(description="Update View", button_style="info", icon="refresh", layout=widgets.Layout(width='140px'))
    out_area = widgets.Output()

    def update_plot_display():
        with out_area:
            clear_output(wait=True)
            if not runs:
                print("No runs currently stored in memory. Add a run using widgets or code below.")
                return
            render_plots(
                show_active=show_active_w.value,
                show_composition=show_comp_w.value,
                show_solvability=show_solv_w.value,
            )

    def on_add_clicked(b):
        with out_area:
            clear_output(wait=True)
            diff = difficulty_w.value
            sol = solver_w.value
            c_count = int(campaigns_w.value)
            mr = int(max_rounds_w.value)
            pr = int(probes_w.value)

            print(f"Running simulation: {diff}/{sol} ({c_count} campaigns, max {mr} rounds)... ")
            flags = RuleFlags(
                scars=True, curses=True, blessings=True, attrition=True,
                sealed_tomb_victory=False, rank_anchor_victory=False
            )
            max_redeals = DIFFICULTIES[diff]
            rng = random.Random()

            campaign_results = []
            for _ in range(c_count):
                res = run_collapse_campaign(
                    rng=rng,
                    max_redeals=max_redeals,
                    flags=flags,
                    max_rounds=mr,
                    solver_name=sol,
                    probe_solver_name='greedy',
                    n_probes=pr,
                    sample_interval=1,
                )
                campaign_results.append(res)

            agg, summary = aggregate_results(campaign_results, mr, 1)
            run_label = f"{diff}/{sol} #{len(runs)+1}"
            runs.append({"label": run_label, "data": agg})
            print(f"Added run '{run_label}'. Total active runs: {len(runs)}")
            render_plots(
                show_active=show_active_w.value,
                show_composition=show_comp_w.value,
                show_solvability=show_solv_w.value,
            )

    def on_clear_clicked(b):
        global runs
        with out_area:
            clear_output(wait=True)
            runs = []
            plt.close('all')
            print("Cleared all stored runs from memory.")

    def on_toggle_changed(change):
        if change['type'] == 'change' and change['name'] == 'value':
            update_plot_display()

    show_active_w.observe(on_toggle_changed, names='value')
    show_comp_w.observe(on_toggle_changed, names='value')
    show_solv_w.observe(on_toggle_changed, names='value')

    btn_add.on_click(on_add_clicked)
    btn_clear.on_click(on_clear_clicked)
    btn_update.on_click(lambda b: update_plot_display())

    display(widgets.VBox([
        difficulty_w,
        solver_w,
        campaigns_w,
        max_rounds_w,
        probes_w,
        widgets.HTML("<b>Toggle Charts:</b>"),
        show_active_w,
        show_comp_w,
        show_solv_w,
        widgets.HBox([btn_add, btn_clear, btn_update]),
        out_area
    ]))
except ImportError:
    print("ipywidgets not installed. Use programmatic cells below.")
